# Caso práctico: Rescate de una fotografía degradada

En este ejercicio aplicarás de forma integrada los conceptos aprendidos en las sesiones anteriores:

- **S1** — Carga, exploración, histogramas, mejora de contraste y bordes
- **S2** — Umbralización con Otsu y limpieza morfológica
- **S3** — Detección de rostros con Haar Cascades

### Escenario

Trabajas en un laboratorio de restauración fotográfica. Un cliente te entrega una foto antigua que está **muy oscura y con ruido**. Tu objetivo es:

1. Analizar el problema
2. Restaurar la imagen lo suficiente para poder detectar el rostro
3. Extraer la región del rostro detectado

Cada ejercicio te guía paso a paso. Completa las celdas de código donde veas `# TU CÓDIGO AQUÍ`.

## Objetivos

Al completar este ejercicio serás capaz de:

1. Explorar una imagen y diagnosticar problemas de calidad mediante histogramas.
2. Aplicar CLAHE para restaurar el contraste.
3. Comparar el efecto del preprocesamiento sobre bordes (Canny) y segmentación (Otsu).
4. Limpiar una máscara binaria con operaciones morfológicas.
5. Usar Haar Cascades para detectar un rostro y extraer su región de interés.
6. Construir un pipeline completo que integre todos los pasos.

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data, color, exposure, filters, morphology
from skimage.filters import threshold_otsu
from skimage.feature import canny

plt.rcParams['figure.figsize'] = (10, 5)

# Ruta a los modelos Haar (sin caracteres especiales)
MODELS_DIR = 'C:/Users/nekov/Downloads/Preparacion_CV1_models/'

## Ejercicio 1 — Carga y exploración básica (S1)

Carga la imagen `data.astronaut()` de scikit-image.

**Tareas:**
1. Guarda la imagen en una variable `img_original`.
2. Imprime: forma (`shape`), tipo de dato (`dtype`), valor mínimo y máximo.
3. Muestra la imagen con `plt.imshow()`.

In [ ]:
# Ejercicio 1

# TU CÓDIGO AQUÍ
img_original = ...

# Imprime información básica
# TU CÓDIGO AQUÍ

# Muestra la imagen
# TU CÓDIGO AQUÍ

## Preparación — Simular la fotografía dañada

La siguiente celda simula una foto antigua: **oscurece** la imagen y le **añade ruido gaussiano**.

> Esta celda está completa — solo ejecútala. Observa cómo cambia la imagen.

In [ ]:
# --- Celda proporcionada (no modificar) ---

# Oscurecer: multiplicar por 0.3 (pierde el 70% de brillo)
img_oscura = (img_original.astype('float64') / 255.0) * 0.3

# Añadir ruido gaussiano
rng = np.random.default_rng(42)
ruido = rng.normal(0, 0.03, img_oscura.shape)
img_degradada = np.clip(img_oscura + ruido, 0, 1)

# Convertir a uint8 para compatibilidad con OpenCV
img_degradada_uint8 = (img_degradada * 255).astype('uint8')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_original)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(img_degradada)
axes[1].set_title('Fotografía degradada (oscura + ruido)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'Original  — rango: [{img_original.min()}, {img_original.max()}], dtype: {img_original.dtype}')
print(f'Degradada — rango: [{img_degradada.min():.3f}, {img_degradada.max():.3f}], dtype: {img_degradada.dtype}')

## Ejercicio 2 — Diagnóstico con histogramas (S1)

Para entender qué le pasa a la imagen degradada, compara los histogramas de la versión original y la degradada.

**Tareas:**
1. Convierte ambas imágenes a escala de grises usando `color.rgb2gray()`.
2. Dibuja los dos histogramas lado a lado usando `axes[i].hist(imagen.ravel(), bins=256)`.
3. **Observa:** ¿en qué rango están concentrados los valores de la imagen degradada?

> **Pista:** `color.rgb2gray()` devuelve valores float64 en [0, 1].

In [ ]:
# Ejercicio 2

# Paso 1: convertir a escala de grises
gray_original  = ...  # TU CÓDIGO AQUÍ
gray_degradada = ...  # TU CÓDIGO AQUÍ

# Paso 2: dibujar histogramas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de la original
# TU CÓDIGO AQUÍ
axes[0].set_title('Histograma — original')
axes[0].set_xlabel('Intensidad')
axes[0].set_ylabel('Frecuencia')

# Histograma de la degradada
# TU CÓDIGO AQUÍ
axes[1].set_title('Histograma — degradada')
axes[1].set_xlabel('Intensidad')

plt.tight_layout()
plt.show()

# Paso 3: ¿en qué rango están los valores de la degradada?
# Escribe tu observación como comentario:
# ...

## Ejercicio 3 — Mejora de contraste con CLAHE (S1)

Aplica ecualización adaptativa (CLAHE) a la imagen degradada para recuperar el contraste.

**Tareas:**
1. Aplica `exposure.equalize_adapthist()` a `gray_degradada` con `clip_limit=0.03`.
2. Muestra tres imágenes lado a lado: original gris, degradada gris, mejorada con CLAHE.
3. Dibuja el histograma de la imagen mejorada y compáralo con el de la degradada.

> **Pista:** `equalize_adapthist` recibe una imagen en escala de grises y devuelve float64 [0,1].

In [ ]:
# Ejercicio 3

# Paso 1: aplicar CLAHE
gray_mejorada = ...  # TU CÓDIGO AQUÍ

# Paso 2: mostrar las tres versiones
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(gray_original, cmap='gray')
axes[0].set_title('Original (gris)')
axes[0].axis('off')

# TU CÓDIGO AQUÍ — mostrar gray_degradada en axes[1] y gray_mejorada en axes[2]

plt.tight_layout()
plt.show()

# Paso 3: comparar histogramas (degradada vs mejorada)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TU CÓDIGO AQUÍ — histograma de gray_degradada en axes[0], gray_mejorada en axes[1]

plt.tight_layout()
plt.show()

## Ejercicio 4 — Detección de bordes: antes y después (S1)

Compara la detección de bordes con **Sobel** y **Canny** sobre la imagen degradada y la mejorada.

**Tareas:**
1. Aplica `filters.sobel()` a `gray_degradada` y a `gray_mejorada`.
2. Aplica `canny()` con `sigma=1.5` a ambas versiones.
3. Muestra los cuatro resultados en una cuadrícula 2×2.
4. **Observa:** ¿en cuál de las dos versiones los bordes del rostro son más nítidos?

In [ ]:
# Ejercicio 4

# Paso 1: Sobel
sobel_degradada = ...  # TU CÓDIGO AQUÍ
sobel_mejorada  = ...  # TU CÓDIGO AQUÍ

# Paso 2: Canny con sigma=1.5
canny_degradada = ...  # TU CÓDIGO AQUÍ
canny_mejorada  = ...  # TU CÓDIGO AQUÍ

# Paso 3: mostrar en cuadrícula 2x2
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# TU CÓDIGO AQUÍ — fila 0: Sobel degradada y mejorada
#                   fila 1: Canny degradada y mejorada

plt.suptitle('Efecto de CLAHE sobre la detección de bordes', fontsize=12)
plt.tight_layout()
plt.show()

# Paso 4: observación
# ...

## Ejercicio 5 — Umbralización con Otsu (S2)

Aplica el método de Otsu para segmentar la imagen en fondo y primer plano.

**Tareas:**
1. Calcula el umbral de Otsu sobre `gray_degradada` y sobre `gray_mejorada` usando `threshold_otsu()`.
2. Crea las máscaras binarias correspondientes (píxeles por encima del umbral = True).
3. Muestra ambas máscaras lado a lado.
4. Imprime el valor del umbral en cada caso.
5. **Observa:** ¿cuál de las dos máscaras separa mejor el primer plano del fondo?

In [ ]:
# Ejercicio 5

# Paso 1: calcular umbrales de Otsu
umbral_degradada = ...  # TU CÓDIGO AQUÍ
umbral_mejorada  = ...  # TU CÓDIGO AQUÍ

print(f'Umbral Otsu (degradada) : {umbral_degradada:.3f}')
print(f'Umbral Otsu (mejorada)  : {umbral_mejorada:.3f}')

# Paso 2: crear máscaras binarias
mascara_degradada = ...  # TU CÓDIGO AQUÍ
mascara_mejorada  = ...  # TU CÓDIGO AQUÍ

# Paso 3: mostrar resultados
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# TU CÓDIGO AQUÍ — mostrar ambas máscaras con imshow y cmap='gray'

plt.tight_layout()
plt.show()

## Ejercicio 6 — Limpieza morfológica (S2)

La máscara de Otsu suele tener agujeros y regiones ruidosas. Limpia la máscara mejorada.

**Tareas:**
1. Aplica **cierre morfológico** (`morphology.closing`) con `morphology.disk(5)` para rellenar agujeros.
2. Elimina **regiones pequeñas** (`morphology.remove_small_objects`) con `min_size=500`.
3. Muestra tres imágenes: máscara original, tras cierre, tras eliminar regiones pequeñas.

In [ ]:
# Ejercicio 6

# Paso 1: cierre morfológico
mascara_cierre = ...  # TU CÓDIGO AQUÍ

# Paso 2: eliminar regiones pequeñas
mascara_limpia = ...  # TU CÓDIGO AQUÍ

# Paso 3: mostrar las tres versiones
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mascara_mejorada, cmap='gray')
axes[0].set_title('Otsu (sin limpiar)')
axes[0].axis('off')

# TU CÓDIGO AQUÍ — mostrar mascara_cierre y mascara_limpia en axes[1] y axes[2]

plt.suptitle('Limpieza morfológica paso a paso', fontsize=12)
plt.tight_layout()
plt.show()

## Ejercicio 7 — Detección de rostros con Haar Cascades (S3)

La prueba de fuego: ¿puede el detector de Haar encontrar el rostro?

**Tareas:**
1. Carga el clasificador Haar (`haarcascade_frontalface_default.xml`) desde `MODELS_DIR`.
2. Detecta rostros en la **imagen degradada** (usa `img_degradada_uint8` → gris con `cv2.cvtColor`).
3. Detecta rostros en la **imagen mejorada** (convierte `gray_mejorada` a uint8 primero).
4. Muestra ambos resultados lado a lado con bounding boxes verdes.
5. **Observa:** ¿en cuál detecta el rostro?

> **Pista:** `detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30,30))`
> **Pista:** Para convertir float [0,1] a uint8: `(imagen * 255).astype('uint8')`

In [ ]:
# Ejercicio 7
import matplotlib.patches as patches

# Paso 1: cargar el clasificador
detector = ...  # TU CÓDIGO AQUÍ

# Paso 2: preparar imágenes en gris uint8
gray_deg_uint8 = cv2.cvtColor(img_degradada_uint8, cv2.COLOR_RGB2GRAY)
gray_mej_uint8 = ...  # TU CÓDIGO AQUÍ — convertir gray_mejorada a uint8

# Paso 3: detectar en ambas
rostros_deg = ...  # TU CÓDIGO AQUÍ
rostros_mej = ...  # TU CÓDIGO AQUÍ

print(f'Rostros en degradada: {len(rostros_deg)}')
print(f'Rostros en mejorada:  {len(rostros_mej)}')

# Paso 4: visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img_degradada)
axes[0].set_title(f'Degradada — {len(rostros_deg)} rostro(s)')
axes[0].axis('off')

axes[1].imshow(img_degradada)
axes[1].set_title(f'Con CLAHE — {len(rostros_mej)} rostro(s)')
axes[1].axis('off')

# TU CÓDIGO AQUÍ — dibujar bounding boxes con patches.Rectangle
# Ejemplo: rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
#          axes[i].add_patch(rect)

plt.tight_layout()
plt.show()

## Ejercicio 8 — Extracción de la región del rostro (S3)

Extrae la región del rostro detectado de la **imagen original en color**.

**Tareas:**
1. Usa las coordenadas `(x, y, w, h)` del primer rostro detectado en la imagen mejorada.
2. Añade un margen de 20 píxeles alrededor (sin salir de los límites).
3. Recorta esa región de `img_original` y muéstrala.

> **Pista:** Recorte con numpy: `imagen[y1:y2, x1:x2]`
> **Pista:** Usa `max(0, ...)` y `min(limite, ...)` para respetar los límites.

In [ ]:
# Ejercicio 8

if len(rostros_mej) > 0:
    x, y, w, h = rostros_mej[0]
    margen = 20

    # Calcular coordenadas con margen (sin salir de la imagen)
    y1 = ...  # TU CÓDIGO AQUÍ
    y2 = ...  # TU CÓDIGO AQUÍ
    x1 = ...  # TU CÓDIGO AQUÍ
    x2 = ...  # TU CÓDIGO AQUÍ

    # Recortar
    rostro_roi = ...  # TU CÓDIGO AQUÍ

    # Mostrar
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_original)
    axes[0].set_title('Imagen original con detección')
    axes[0].axis('off')
    axes[0].add_patch(patches.Rectangle((x, y), w, h,
                       linewidth=2, edgecolor='lime', facecolor='none'))

    # TU CÓDIGO AQUÍ — mostrar rostro_roi en axes[1]

    plt.tight_layout()
    plt.show()
else:
    print('No se detectó ningún rostro. Revisa los pasos anteriores.')

## Ejercicio 9 — Pipeline completo (S1 + S2 + S3)

Integra todo en una función que reciba cualquier imagen (posiblemente degradada) y devuelva los rostros detectados como recortes.

**Tareas:**
1. Completa la función `detectar_rostros(imagen_rgb)` siguiendo los pasos indicados.
2. Prueba la función con `img_original` y con `img_degradada_uint8`.

> Esta función resume todo el pipeline del curso: **carga → preprocesamiento → detección → extracción**.

In [ ]:
# Ejercicio 9

def detectar_rostros(imagen_rgb, clip_limit=0.03, margen=20):
    """
    Pipeline completo: preprocesa la imagen y devuelve recortes de los rostros.

    Parámetros:
        imagen_rgb  : imagen RGB en uint8 [0,255]
        clip_limit  : parámetro de CLAHE
        margen      : píxeles extra alrededor de cada rostro

    Retorna:
        lista de arrays numpy (recortes de rostros en RGB)
    """
    # 1. Convertir a escala de grises float
    # TU CÓDIGO AQUÍ

    # 2. Aplicar CLAHE
    # TU CÓDIGO AQUÍ

    # 3. Convertir a uint8 para OpenCV
    # TU CÓDIGO AQUÍ

    # 4. Cargar detector y ejecutar detectMultiScale
    # TU CÓDIGO AQUÍ

    # 5. Recortar cada rostro con margen
    rostros_roi = []
    # TU CÓDIGO AQUÍ

    return rostros_roi

# --- Probar ---
print('=== Imagen original ===')
rois_original = detectar_rostros(img_original)
print(f'Rostros encontrados: {len(rois_original)}')

print('\n=== Imagen degradada ===')
rois_degradada = detectar_rostros(img_degradada_uint8)
print(f'Rostros encontrados: {len(rois_degradada)}')

# Mostrar todos los rostros extraídos
todos = rois_original + rois_degradada
if todos:
    fig, axes = plt.subplots(1, len(todos), figsize=(5 * len(todos), 5))
    if len(todos) == 1:
        axes = [axes]
    for ax, roi in zip(axes, todos):
        ax.imshow(roi)
        ax.set_title(f'{roi.shape[1]}x{roi.shape[0]} px')
        ax.axis('off')
    plt.suptitle('Rostros extraídos por el pipeline', fontsize=12)
    plt.tight_layout()
    plt.show()

## Conclusión

En este ejercicio has aplicado de forma integrada el pipeline completo de visión por computador:

```
Imagen degradada (oscura + ruido)
    │
S1: Exploración (shape, dtype, histograma) → Diagnóstico: bajo contraste
    │
S1: Mejora de contraste (CLAHE) → Histograma redistribuido
    │
S1: Bordes (Sobel, Canny) → Verificación: bordes más claros tras CLAHE
    │
S2: Umbralización (Otsu) → Máscara binaria
    │
S2: Limpieza morfológica (cierre + remove_small_objects) → Máscara limpia
    │
S3: Detección (Haar Cascade) → Bounding boxes
    │
S3: Extracción de ROI → Rostro recortado
```

### Lecciones clave

1. **El preprocesamiento importa**: sin CLAHE, el detector de rostros falla en imágenes oscuras.
2. **Cada herramienta tiene su lugar**: histogramas para diagnosticar, CLAHE para corregir, Otsu para segmentar, Haar para detectar.
3. **El pipeline es secuencial**: cada paso depende del anterior y lo mejora.